### Step 1: Imports and Parameters

In [ ]:
### Import Programmes

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
from sklearn.cluster import KMeans
from math import ceil

In [ ]:
# Model parameters
planning_days = 365
m = 8
q = 2
c_b = 5000
c_h = 1000
c_m = 500
c_c = 0.42
lambda_ = 0.012
r_min = 10
r_max = 175

### Step 2: Robot Subset 

In [ ]:
ranges = pd.read_csv("range_scenarios.csv")
locations = pd.read_csv("robot_locations.csv")

locations = locations.rename(columns={"index": "robot_id"})

first_col = ranges.columns[0]
ranges = ranges.rename(columns={first_col: "robot_id"})

robots = (
    locations.merge(ranges, on="robot_id", how="inner")
    .sort_values("robot_id")
    .reset_index(drop=True)
)

scenario_cols = [c for c in robots.columns if c.startswith("s")]

display(robots.head())
print("Shape:", robots.shape)
print("Number of scenario columns:", len(scenario_cols))


In [ ]:
# Random sample of robots
# Choose subset sizes
n_robots = 500
n_scenarios = 50
random_seed = 1

rng = np.random.default_rng(random_seed)

all_scenario_cols = [c for c in robots.columns if c.startswith("s")]

# Sample which scenario columns to keep
selected_scenarios = sorted(
    rng.choice(all_scenario_cols, size=n_scenarios, replace=False).tolist()
)

# Sample robots and keep only the chosen scenarios
robots_subset = (
    robots
    .sample(n=n_robots, random_state=random_seed)
    .sort_values("robot_id")
    .reset_index(drop=True)
    [["robot_id", "longitude", "latitude"] + selected_scenarios]
    .copy()
)

# Mean range across the selected scenarios
robots_subset["mean_range"] = robots_subset[selected_scenarios].mean(axis=1)

display(robots_subset.head())
print("Number of robots kept:", len(robots_subset))
print("Number of scenarios kept:", len(selected_scenarios))
print("Selected scenarios:", selected_scenarios)


### Step 3: Candidate Selection

Weighted K-means clustering using the mean of chosen samples. 

In [ ]:
coords = robots_subset[["longitude", "latitude"]].to_numpy()

# Robots with lower expected range are more important when placing stations.
weights = 1.0 / np.maximum(robots_subset["mean_range"].to_numpy(), 1e-6)

candidate_rows = []
candidate_id = 0

group_sizes = range(1, 17)

for s in group_sizes:
    k = ceil(len(robots_subset) / s)

    kmeans = KMeans(
        n_clusters=k,
        random_state=random_seed,
        n_init=10
    )

    kmeans.fit(coords, sample_weight=weights)

    labels = kmeans.labels_
    centers = kmeans.cluster_centers_

    for cluster_id, center in enumerate(centers):
        cluster_size = int((labels == cluster_id).sum())

        candidate_rows.append({
            "candidate_id": candidate_id,
            "source_group_size": s,
            "cluster_size": cluster_size,
            "longitude": center[0],
            "latitude": center[1]
        })

        candidate_id += 1

candidate_locations = pd.DataFrame(candidate_rows)

# Deduplicate nearby candidate locations
candidate_locations["lon_r"] = candidate_locations["longitude"].round(4)
candidate_locations["lat_r"] = candidate_locations["latitude"].round(4)

candidate_locations = (
    candidate_locations
    .sort_values(["source_group_size", "cluster_size"], ascending=[True, False])
    .drop_duplicates(["lon_r", "lat_r"])
    .drop(columns=["lon_r", "lat_r"])
    .reset_index(drop=True)
)

candidate_locations["candidate_id"] = range(len(candidate_locations))

display(candidate_locations.head())
print("Number of candidate locations:", len(candidate_locations))


In [ ]:
print("Number of candidate locations:", len(candidate_locations))
print("Candidates before deduplication:", len(pd.DataFrame(candidate_rows)))
print("Candidates after deduplication:", len(candidate_locations))
print(candidate_locations["source_group_size"].value_counts().sort_index())


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Plot robot locations
ax.scatter(
    robots_subset["longitude"],
    robots_subset["latitude"],
    c="red",
    s=30,
    alpha=0.6,
    label="Robots"
)

# Plot candidate locations
ax.scatter(
    candidate_locations["longitude"],
    candidate_locations["latitude"],
    c="blue",
    s=70,
    marker="X",
    alpha=0.9,
    label="Candidate locations"
)

ax.set_title("Robot Locations and Generated Candidate Locations")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, alpha=0.3)
ax.legend()
plt.xticks(rotation=90)
plt.tight_layout()
fig.savefig("Graphs/candidates_Q2_500.png", dpi=300, bbox_inches="tight")
plt.show()


### Step 4: Stochastic Scenarios

- uses range scenarios from the chosen sample scenarios
- For each robot in each sampled scenario, samples whether the robot needs charging uaing
$$
p_i = \exp\left(-\lambda^2 (r_i - r_{\min})^2 \right)
$$
- Produces a scenario table with one row per (scenario, robot)

This is a Sample Average Approximation (SAA) approach.  

We do not enumerate all possible charging combinations, as that would be computationally infeasible due to the problem size.

In [ ]:
# Convert the selected scenario columns into long format
range_long = robots_subset.melt(
    id_vars=["robot_id", "longitude", "latitude", "mean_range"],
    value_vars=selected_scenarios,
    var_name="range_scenario",
    value_name="scenario_range"
).copy()

# Probability that a robot needs charging in a scenario
range_long["p_charge"] = np.exp(
    -(lambda_ ** 2) * (range_long["scenario_range"] - r_min) ** 2
)

# Sample whether charging is needed
range_long["needs_charge"] = rng.binomial(
    1,
    range_long["p_charge"]
)

# Create a cleaner integer scenario index as well
scenario_id_map = {
    s: idx for idx, s in enumerate(selected_scenarios)
}

range_long["scenario_id"] = range_long["range_scenario"].map(scenario_id_map)

scenario_table = range_long[
    ["scenario_id", "range_scenario", "robot_id", "scenario_range", "p_charge", "needs_charge"]
].copy()

display(scenario_table.head())
print("Number of sampled scenarios:", scenario_table["scenario_id"].nunique())
print("Number of robot-scenario rows:", len(scenario_table))


### Step 5: Expected Assignment Costs


For each robot-station pair, the model:

1. Compute physical distance  
   - Calculate \( d_{ij} \)

2. Evaluate across sampled scenarios 
   - Look across all sampled scenarios

3. Per-scenario evaluation  
   For each scenario:
   - Get the robot range
   - Check whether the station is reachable
   - If the robot needs charging, compute charging cost
   - If the station is unreachable and charging is needed, include transport cost

4. Aggregate results  
   - Average costs across all scenarios



 The output is a expected-cost table, used by the heuristic and local search.

In [ ]:
robot_ids = robots_subset["robot_id"].astype(int).tolist()
candidate_ids = candidate_locations["candidate_id"].astype(int).tolist()

# Coordinates
robot_xy = robots_subset[["longitude", "latitude"]].to_numpy()
candidate_xy = candidate_locations[["longitude", "latitude"]].to_numpy()

# Distance matrix: rows = robots, columns = candidates
distance_matrix = np.sqrt(
    ((robot_xy[:, None, :] - candidate_xy[None, :, :]) ** 2).sum(axis=2)
)

# Scenario data in matrix form:
# rows = robots, columns = sampled scenarios
scenario_range_matrix = (
    scenario_table
    .pivot(index="robot_id", columns="scenario_id", values="scenario_range")
    .loc[robot_ids]
    .to_numpy()
)

needs_charge_matrix = (
    scenario_table
    .pivot(index="robot_id", columns="scenario_id", values="needs_charge")
    .loc[robot_ids]
    .to_numpy()
)

distance_records = []

for i_idx, robot_id in enumerate(robot_ids): # loop over every (robot, station) pair
    for j_idx, candidate_id in enumerate(candidate_ids):
        d_ij = float(distance_matrix[i_idx, j_idx])

        scenario_ranges = scenario_range_matrix[i_idx, :]
        needs_charge = needs_charge_matrix[i_idx, :]

        # check which sampled scenarios the robot is out of range for candidate j
        unreachable = (d_ij > scenario_ranges).astype(int) 

        charging_cost_scen = planning_days * c_c * (
            (r_max - scenario_ranges + d_ij) * (1 - unreachable)
            + (r_max - scenario_ranges) * unreachable
        ) * needs_charge

        transport_cost_scen = planning_days * c_h * unreachable * needs_charge

        total_cost_scen = charging_cost_scen + transport_cost_scen

        distance_records.append({
            "robot_id": int(robot_id),
            "candidate_id": int(candidate_id),
            "d_ij": d_ij,
            "expected_charging_cost": float(charging_cost_scen.mean()),
            "expected_transport_cost": float(transport_cost_scen.mean()),
            "assign_cost": float(total_cost_scen.mean()),
            "expected_needs_charge": float(needs_charge.mean()),
            "expected_unreachable": float((unreachable * needs_charge).mean())
        })

stochastic_distance_table = pd.DataFrame(distance_records)

display(stochastic_distance_table.head())
print("Robot-station pairs:", len(stochastic_distance_table))


### Step 6: Greedy Heuristic

- Run the greedy heuristic using expected assignment costs  
- Build an initial feasible solution  


In [ ]:
### Evaluate a set of open stations under conservative charger limits (no more assignments than there are chargers)

def evaluate_station_set_conservative(
    open_station_set,
    robot_ids,
    stochastic_distance_table,
    candidate_locations,
    q,
    m,
    c_b,
    c_m
):
    open_station_set = set(open_station_set)

    if len(open_station_set) == 0:
        return None

    # Keep only rows for currently open stations
    dt = stochastic_distance_table[
        stochastic_distance_table["candidate_id"].isin(open_station_set)
    ].copy()

    if dt.empty:
        return None

    # Maximum robots a station can take
    station_capacity = {
        j: q * m for j in open_station_set
    }

    # Sort all robot-station options by expected assignment cost
    dt = dt.sort_values(["assign_cost", "robot_id", "candidate_id"]).reset_index(drop=True)

    assigned_robots = set()
    station_load = {j: 0 for j in open_station_set}
    assignment_rows = []

    # Greedy assignment of robots to cheapest open station with spare capacity
    for row in dt.itertuples(index=False):
        i = int(row.robot_id)
        j = int(row.candidate_id)

        if i in assigned_robots:
            continue

        if station_load[j] >= station_capacity[j]:
            continue

        assigned_robots.add(i)
        station_load[j] += 1

        assignment_rows.append({
            "robot_id": i,
            "candidate_id": j,
            "d_ij": float(row.d_ij),
            "expected_charging_cost": float(row.expected_charging_cost),
            "expected_transport_cost": float(row.expected_transport_cost),
            "assign_cost": float(row.assign_cost),
            "expected_needs_charge": float(row.expected_needs_charge),
            "expected_unreachable": float(row.expected_unreachable)
        })

        if len(assigned_robots) == len(robot_ids):
            break

    # If not all robots could be assigned, this station set is infeasible
    if len(assigned_robots) < len(robot_ids):
        return None

    assignments_df = pd.DataFrame(assignment_rows)

    # Build station summary and charger counts
    station_rows = []

    for j in sorted(open_station_set):
        robots_assigned = int((assignments_df["candidate_id"] == j).sum())
        chargers = int(ceil(robots_assigned / q)) if robots_assigned > 0 else 1

        if chargers > m:
            return None

        station_rows.append({
            "candidate_id": j,
            "robots_assigned": robots_assigned,
            "chargers": chargers
        })

    stations_df = pd.DataFrame(station_rows).merge(
        candidate_locations[["candidate_id", "longitude", "latitude"]],
        on="candidate_id",
        how="left"
    )

    buildings_cost = len(open_station_set) * c_b + stations_df["chargers"].sum() * c_m
    assignment_cost = assignments_df["expected_charging_cost"].sum()
    transport_cost = assignments_df["expected_transport_cost"].sum()
    total_cost = buildings_cost + assignment_cost + transport_cost

    return {
        "open_station_set": open_station_set,
        "stations_df": stations_df,
        "assignments_df": assignments_df,
        "buildings_cost": buildings_cost,
        "assignment_cost": assignment_cost,
        "transport_cost": transport_cost,
        "total_cost": total_cost
    }


At each iteration:

1) test opening each currently closed candidate station
2) evaluate the resulting full assignment
3) pick the opening that gives the lowest total cost
4) stop when no opening improves the current solution


In [ ]:
# Greedy Construction Heuristic

# Sets and candidate IDs
robot_ids = robots_subset["robot_id"].astype(int).tolist()
candidate_ids = candidate_locations["candidate_id"].astype(int).tolist()

# Expected assignment-cost lookup
assign_cost = {
    (int(row.robot_id), int(row.candidate_id)): float(row.assign_cost)
    for row in stochastic_distance_table.itertuples(index=False)
}

# Optional extra lookups for reporting
expected_charging_cost_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.expected_charging_cost)
    for row in stochastic_distance_table.itertuples(index=False)
}

expected_transport_cost_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.expected_transport_cost)
    for row in stochastic_distance_table.itertuples(index=False)
}

expected_unreachable_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.expected_unreachable)
    for row in stochastic_distance_table.itertuples(index=False)
}

distance_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.d_ij)
    for row in stochastic_distance_table.itertuples(index=False)
}

# Maintain heuristic state
unassigned_robots = set(robot_ids)
open_stations = []
station_chargers = {}
robot_assignment = {}

iteration = 0

while unassigned_robots:
    iteration += 1
    best_choice = None
    best_avg_cost = float("inf")

    # loop over all unopened candidate stations
    for j in candidate_ids:
        if j in open_stations:
            continue

        # expected costs of assigning each currently unassigned robot to station j
        robot_costs = sorted(
            [(i, assign_cost[(i, j)]) for i in unassigned_robots],
            key=lambda x: x[1]
        )

        for z in range(1, m + 1):
            capacity = q * z
            chosen = robot_costs[:capacity]

            if len(chosen) == 0:
                continue

            total_cost = c_b + c_m * z + sum(cost for _, cost in chosen)
            avg_cost = total_cost / len(chosen)

            if avg_cost < best_avg_cost:
                best_avg_cost = avg_cost
                best_choice = {
                    "station": j,
                    "chargers": z,
                    "robots": [i for i, _ in chosen],
                    "total_cost": total_cost,
                    "avg_cost": avg_cost
                }

    if best_choice is None:
        raise RuntimeError("No feasible stochastic greedy station choice found.")

    chosen_station = best_choice["station"]
    chosen_chargers = best_choice["chargers"]
    chosen_robots = best_choice["robots"]

    open_stations.append(chosen_station)
    station_chargers[chosen_station] = chosen_chargers

    for i in chosen_robots:
        robot_assignment[i] = chosen_station
        unassigned_robots.remove(i)

    print(
        f"Iteration {iteration}: opened station {chosen_station} "
        f"with {chosen_chargers} chargers, assigned {len(chosen_robots)} robots, "
        f"{len(unassigned_robots)} robots remaining"
    )

print("Stochastic greedy construction completed.")
print("Opened stations:", open_stations)
print("Chargers per station:", station_chargers)


In [ ]:
# Build output tables from the stochastic greedy heuristic

heuristic_stations_df = pd.DataFrame([
    {
        "candidate_id": j,
        "chargers": station_chargers[j]
    }
    for j in open_stations
]).merge(
    candidate_locations[["candidate_id", "longitude", "latitude"]],
    on="candidate_id",
    how="left"
)

heuristic_stations_df["robots_assigned"] = heuristic_stations_df["candidate_id"].map(
    pd.Series(robot_assignment).value_counts()
).fillna(0).astype(int)

heuristic_assignments_df = pd.DataFrame([
    {
        "robot_id": i,
        "candidate_id": j,
        "d_ij": distance_lookup[(i, j)],
        "expected_charging_cost": expected_charging_cost_lookup[(i, j)],
        "expected_transport_cost": expected_transport_cost_lookup[(i, j)],
        "assign_cost": assign_cost[(i, j)],
        "expected_unreachable": expected_unreachable_lookup[(i, j)]
    }
    for i, j in robot_assignment.items()
]).sort_values("robot_id").reset_index(drop=True)

heuristic_buildings_cost = sum(c_b + c_m * station_chargers[j] for j in open_stations)
heuristic_assignment_cost = heuristic_assignments_df["expected_charging_cost"].sum()
heuristic_transport_cost = heuristic_assignments_df["expected_transport_cost"].sum()
heuristic_total_cost = (
    heuristic_buildings_cost
    + heuristic_assignment_cost
    + heuristic_transport_cost
)

print("Heuristic total cost:", heuristic_total_cost)

display(heuristic_stations_df)
display(heuristic_assignments_df.head())


### Step 6.5: Optional MIP optimisation if not doing local search

In [ ]:
# Step 6.5: MIP after the heuristic open-station set
# Fixed open stations, fixed robot-to-station assignment across scenarios, expected costs from stochastic_distance_table.

J_heuristic_mip = sorted(heuristic_stations_df["candidate_id"].astype(int).tolist())
heuristic_mip_robot_ids = robots_subset["robot_id"].astype(int).tolist()
heuristic_mip_scenario_ids = sorted(scenario_table["scenario_id"].unique())

heuristic_mip_dt_open = stochastic_distance_table[
    stochastic_distance_table["candidate_id"].isin(J_heuristic_mip)
].copy()

heuristic_mip_expected_cost_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.assign_cost)
    for row in heuristic_mip_dt_open.itertuples(index=False)
}
heuristic_mip_expected_charging_cost_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.expected_charging_cost)
    for row in heuristic_mip_dt_open.itertuples(index=False)
}
heuristic_mip_expected_transport_cost_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.expected_transport_cost)
    for row in heuristic_mip_dt_open.itertuples(index=False)
}
heuristic_mip_expected_unreachable_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.expected_unreachable)
    for row in heuristic_mip_dt_open.itertuples(index=False)
}
heuristic_mip_expected_needs_charge_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.expected_needs_charge)
    for row in heuristic_mip_dt_open.itertuples(index=False)
}
heuristic_mip_distance_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.d_ij)
    for row in heuristic_mip_dt_open.itertuples(index=False)
}

heuristic_mip_model = pyo.ConcreteModel()
heuristic_mip_model.I = pyo.Set(initialize=heuristic_mip_robot_ids)
heuristic_mip_model.J = pyo.Set(initialize=J_heuristic_mip)
heuristic_mip_model.assign_cost = pyo.Param(
    heuristic_mip_model.I,
    heuristic_mip_model.J,
    initialize=heuristic_mip_expected_cost_lookup
)

heuristic_mip_model.z = pyo.Var(heuristic_mip_model.J, domain=pyo.NonNegativeIntegers)
heuristic_mip_model.x = pyo.Var(heuristic_mip_model.I, heuristic_mip_model.J, domain=pyo.Binary)

# Each robot is assigned to exactly one of the fixed open stations.
def heuristic_mip_assign_once_rule(model, i):
    return sum(model.x[i, j] for j in model.J) == 1

heuristic_mip_model.assign_once = pyo.Constraint(
    heuristic_mip_model.I,
    rule=heuristic_mip_assign_once_rule
)

# A station with z[j] chargers can serve at most q*z[j] assigned robots.
def heuristic_mip_charger_capacity_rule(model, j):
    return sum(model.x[i, j] for i in model.I) <= q * model.z[j]

heuristic_mip_model.charger_capacity = pyo.Constraint(
    heuristic_mip_model.J,
    rule=heuristic_mip_charger_capacity_rule
)

# Charger bounds at fixed open stations.
def heuristic_mip_charger_limit_rule(model, j):
    return model.z[j] <= m

heuristic_mip_model.charger_limit = pyo.Constraint(
    heuristic_mip_model.J,
    rule=heuristic_mip_charger_limit_rule
)

def heuristic_mip_min_charger_rule(model, j):
    return model.z[j] >= 1

heuristic_mip_model.min_charger = pyo.Constraint(
    heuristic_mip_model.J,
    rule=heuristic_mip_min_charger_rule
)

# Objective: fixed station building cost + charger cost + expected assignment cost.
def heuristic_mip_objective_rule(model):
    buildings_cost = len(J_heuristic_mip) * c_b + sum(c_m * model.z[j] for j in model.J)
    expected_assignment_cost = sum(
        model.assign_cost[i, j] * model.x[i, j]
        for i in model.I
        for j in model.J
    )
    return buildings_cost + expected_assignment_cost

heuristic_mip_model.obj = pyo.Objective(
    rule=heuristic_mip_objective_rule,
    sense=pyo.minimize
)

heuristic_mip_solver = pyo.SolverFactory("gurobi")
heuristic_mip_solver.options["TimeLimit"] = 300
heuristic_mip_results = heuristic_mip_solver.solve(heuristic_mip_model, tee=True)

heuristic_mip_stations_df = pd.DataFrame([
    {
        "candidate_id": int(j),
        "chargers": int(round(pyo.value(heuristic_mip_model.z[j])))
    }
    for j in heuristic_mip_model.J
]).merge(
    candidate_locations[["candidate_id", "longitude", "latitude"]],
    on="candidate_id",
    how="left"
)

heuristic_mip_assignment_rows = []
for i in heuristic_mip_model.I:
    for j in heuristic_mip_model.J:
        if pyo.value(heuristic_mip_model.x[i, j]) > 0.5:
            heuristic_mip_assignment_rows.append({
                "robot_id": int(i),
                "candidate_id": int(j),
                "d_ij": heuristic_mip_distance_lookup[(i, j)],
                "expected_needs_charge": heuristic_mip_expected_needs_charge_lookup[(i, j)],
                "expected_unreachable": heuristic_mip_expected_unreachable_lookup[(i, j)],
                "expected_charging_cost": heuristic_mip_expected_charging_cost_lookup[(i, j)],
                "expected_transport_cost": heuristic_mip_expected_transport_cost_lookup[(i, j)],
                "assign_cost": heuristic_mip_expected_cost_lookup[(i, j)]
            })

heuristic_mip_assignments_df = (
    pd.DataFrame(heuristic_mip_assignment_rows)
    .sort_values("robot_id")
    .reset_index(drop=True)
)

heuristic_mip_stations_df["robots_assigned"] = heuristic_mip_stations_df["candidate_id"].map(
    heuristic_mip_assignments_df["candidate_id"].value_counts()
).fillna(0).astype(int)

heuristic_mip_buildings_cost = len(J_heuristic_mip) * c_b + sum(
    c_m * pyo.value(heuristic_mip_model.z[j])
    for j in heuristic_mip_model.J
)
heuristic_mip_expected_assignment_cost = sum(
    pyo.value(heuristic_mip_model.assign_cost[i, j]) * pyo.value(heuristic_mip_model.x[i, j])
    for i in heuristic_mip_model.I
    for j in heuristic_mip_model.J
)
heuristic_mip_expected_charging_cost = heuristic_mip_assignments_df["expected_charging_cost"].sum()
heuristic_mip_expected_transport_cost = heuristic_mip_assignments_df["expected_transport_cost"].sum()
heuristic_mip_expected_transport_robots = heuristic_mip_assignments_df["expected_unreachable"].sum()
heuristic_mip_total_cost = pyo.value(heuristic_mip_model.obj)

heuristic_mip_summary_df = pd.DataFrame([
    {"Solution": "MIP after heuristic", "Metric": "Number of robots", "Value": f"{len(heuristic_mip_robot_ids):,.0f}"},
    {"Solution": "MIP after heuristic", "Metric": "Number of scenarios used for expectations", "Value": f"{len(heuristic_mip_scenario_ids):,.0f}"},
    {"Solution": "MIP after heuristic", "Metric": "Number of stations", "Value": f"{len(J_heuristic_mip):,.0f}"},
    {"Solution": "MIP after heuristic", "Metric": "Number of chargers", "Value": f"{int(round(sum(pyo.value(heuristic_mip_model.z[j]) for j in heuristic_mip_model.J))):,.0f}"},
    {"Solution": "MIP after heuristic", "Metric": "Total cost", "Value": f"{heuristic_mip_total_cost:,.2f}"},
    {"Solution": "MIP after heuristic", "Metric": "Buildings cost", "Value": f"{heuristic_mip_buildings_cost:,.2f}"},
    {"Solution": "MIP after heuristic", "Metric": "Expected assignment cost", "Value": f"{heuristic_mip_expected_assignment_cost:,.2f}"},
    {"Solution": "MIP after heuristic", "Metric": "Expected charging cost", "Value": f"{heuristic_mip_expected_charging_cost:,.2f}"},
    {"Solution": "MIP after heuristic", "Metric": "Expected transport cost", "Value": f"{heuristic_mip_expected_transport_cost:,.2f}"},
    {"Solution": "MIP after heuristic", "Metric": "Expected robots needing transport", "Value": f"{heuristic_mip_expected_transport_robots:,.2f}"},
])

heuristic_mip_solution = {
    "name": "MIP after heuristic",
    "model": heuristic_mip_model,
    "results": heuristic_mip_results,
    "open_station_set": set(J_heuristic_mip),
    "stations_df": heuristic_mip_stations_df,
    "assignments_df": heuristic_mip_assignments_df,
    "summary_df": heuristic_mip_summary_df,
    "total_cost": heuristic_mip_total_cost,
    "buildings_cost": heuristic_mip_buildings_cost,
    "expected_assignment_cost": heuristic_mip_expected_assignment_cost,
    "expected_recourse_cost": heuristic_mip_expected_assignment_cost,
    "expected_charging_cost": heuristic_mip_expected_charging_cost,
    "expected_transport_cost": heuristic_mip_expected_transport_cost,
    "expected_transport_robots": heuristic_mip_expected_transport_robots,
}

print("MIP after heuristic objective value:", f"{heuristic_mip_total_cost:,.2f}")
display(heuristic_mip_stations_df)
display(heuristic_mip_assignments_df.head())
display(heuristic_mip_summary_df)

### Step 7: Local Search 

Apply neighbourhood search using the same moves as in 1c\_a:

- Swap stations  
- Open stations  
- Close stations  

Each move is evaluated using expected costs.

In [ ]:
# Evaluate an open station set under the stochastic heuristic model

candidate_xy = candidate_locations.set_index("candidate_id")[["longitude", "latitude"]]
all_candidates = set(candidate_locations["candidate_id"].astype(int).tolist())
robot_ids = robots_subset["robot_id"].astype(int).tolist()

def evaluate_station_set_stochastic_greedy(
    open_station_set,
    robot_ids,
    stochastic_distance_table,
    candidate_locations,
    q,
    m,
    c_b,
    c_m
):
    open_station_set = set(open_station_set)

    if len(open_station_set) == 0:
        return None

    dt = stochastic_distance_table[
        stochastic_distance_table["candidate_id"].isin(open_station_set)
    ].copy()

    if dt.empty:
        return None

    # For each robot, rank open stations by expected assignment cost
    dt = dt.sort_values(["robot_id", "assign_cost", "candidate_id"]).reset_index(drop=True)

    station_load = {j: 0 for j in open_station_set}
    station_capacity = {j: q * m for j in open_station_set}
    assignment_rows = []

    # Greedily assign each robot to the cheapest feasible open station
    for i in robot_ids:
        robot_options = dt[dt["robot_id"] == i]

        assigned = False
        for row in robot_options.itertuples(index=False):
            j = int(row.candidate_id)

            if station_load[j] < station_capacity[j]:
                station_load[j] += 1
                assignment_rows.append({
                    "robot_id": int(row.robot_id),
                    "candidate_id": j,
                    "d_ij": float(row.d_ij),
                    "expected_charging_cost": float(row.expected_charging_cost),
                    "expected_transport_cost": float(row.expected_transport_cost),
                    "assign_cost": float(row.assign_cost),
                    "expected_needs_charge": float(row.expected_needs_charge),
                    "expected_unreachable": float(row.expected_unreachable)
                })
                assigned = True
                break

        if not assigned:
            return None

    assignments_df = pd.DataFrame(assignment_rows)

    station_rows = []
    for j in sorted(open_station_set):
        robots_assigned = int((assignments_df["candidate_id"] == j).sum())
        chargers = int(ceil(robots_assigned / q)) if robots_assigned > 0 else 1

        if chargers > m:
            return None

        station_rows.append({
            "candidate_id": j,
            "robots_assigned": robots_assigned,
            "chargers": chargers
        })

    stations_df = pd.DataFrame(station_rows).merge(
        candidate_locations[["candidate_id", "longitude", "latitude"]],
        on="candidate_id",
        how="left"
    )

    buildings_cost = len(open_station_set) * c_b + stations_df["chargers"].sum() * c_m
    assignment_cost = assignments_df["expected_charging_cost"].sum()
    transport_cost = assignments_df["expected_transport_cost"].sum()
    total_cost = buildings_cost + assignment_cost + transport_cost

    return {
        "open_station_set": open_station_set,
        "stations_df": stations_df,
        "assignments_df": assignments_df,
        "buildings_cost": buildings_cost,
        "assignment_cost": assignment_cost,
        "transport_cost": transport_cost,
        "total_cost": total_cost
    }


In [ ]:
### Set the initial solution from the stochastic greedy heuristic

current_station_set = set(heuristic_stations_df["candidate_id"].astype(int).tolist())

current_solution = evaluate_station_set_stochastic_greedy(
    open_station_set=current_station_set,
    robot_ids=robot_ids,
    stochastic_distance_table=stochastic_distance_table,
    candidate_locations=candidate_locations,
    q=q,
    m=m,
    c_b=c_b,
    c_m=c_m
)

if current_solution is None:
    raise RuntimeError("Initial greedy stochastic solution could not be evaluated.")

print("Starting greedy stochastic cost:", current_solution["total_cost"])


In [ ]:
### Swap Neighborhood Search
def generate_swap_neighbours_guided(open_set, candidate_locations, nearest_k=10):
    closed_set = all_candidates - open_set
    neighbours = []

    for j_out in open_set:
        out_xy = candidate_xy.loc[j_out].to_numpy(dtype=float)

        candidate_distances = []
        for j_in in closed_set:
            in_xy = candidate_xy.loc[j_in].to_numpy(dtype=float)
            dist = np.linalg.norm(out_xy - in_xy)
            candidate_distances.append((j_in, dist))

        candidate_distances.sort(key=lambda x: x[1])

        for j_in, _ in candidate_distances[:nearest_k]:
            neighbour = (open_set - {j_out}) | {j_in}
            neighbours.append(("swap", j_out, j_in, neighbour))

    return neighbours


In [ ]:
### Open Neighborhood Search

def generate_open_neighbours_guided(
    current_solution,
    stochastic_distance_table,
    top_robot_count=10,
    nearest_candidates_per_robot=5
):
    open_set = current_solution["open_station_set"]
    closed_set = all_candidates - open_set

    assignments_df = current_solution["assignments_df"]

    expensive_robots = (
        assignments_df
        .sort_values("assign_cost", ascending=False)
        .head(top_robot_count)["robot_id"]
        .astype(int)
        .tolist()
    )

    proposed_candidates = set()

    for i in expensive_robots:
        robot_options = stochastic_distance_table[
            (stochastic_distance_table["robot_id"] == i) &
            (stochastic_distance_table["candidate_id"].isin(closed_set))
        ].sort_values("assign_cost")

        for j in robot_options.head(nearest_candidates_per_robot)["candidate_id"].astype(int).tolist():
            proposed_candidates.add(j)

    neighbours = []
    for j_in in proposed_candidates:
        neighbour = open_set | {j_in}
        neighbours.append(("open", None, j_in, neighbour))

    return neighbours


In [ ]:
### Close Neighborhood Search

def generate_close_neighbours_guided(current_solution, min_open=1, max_close_candidates=5):
    open_set = current_solution["open_station_set"]
    assignments_df = current_solution["assignments_df"]

    if len(open_set) <= min_open:
        return []

    station_scores = (
        assignments_df.groupby("candidate_id")
        .agg(
            robots_assigned=("robot_id", "count"),
            total_assign_cost=("assign_cost", "sum")
        )
        .reset_index()
        .sort_values(["robots_assigned", "total_assign_cost"], ascending=[True, False])
    )

    close_candidates = station_scores.head(max_close_candidates)["candidate_id"].astype(int).tolist()

    neighbours = []
    for j_out in close_candidates:
        if len(open_set) - 1 >= min_open:
            neighbour = open_set - {j_out}
            neighbours.append(("close", j_out, None, neighbour))

    return neighbours


In [ ]:
#### Find first improving neighbour

import time

def find_first_improving_neighbour_stochastic(
    current_solution,
    stochastic_distance_table,
    candidate_locations,
    q,
    m,
    c_b,
    c_m,
    swap_nearest_k=10,
    top_robot_count=10,
    open_nearest_candidates_per_robot=5,
    max_close_candidates=5,
    allow_open=True,
    allow_close=True,
    global_start_time=None,
    time_limit=None
):
    current_cost = current_solution["total_cost"]

    neighbour_specs = []
    neighbour_specs += generate_swap_neighbours_guided(
        current_solution["open_station_set"],
        candidate_locations,
        nearest_k=swap_nearest_k
    )

    if allow_open:
        neighbour_specs += generate_open_neighbours_guided(
            current_solution,
            stochastic_distance_table,
            top_robot_count=top_robot_count,
            nearest_candidates_per_robot=open_nearest_candidates_per_robot
        )

    if allow_close:
        neighbour_specs += generate_close_neighbours_guided(
            current_solution,
            min_open=1,
            max_close_candidates=max_close_candidates
        )

    for move_type, j_out, j_in, neighbour_set in neighbour_specs:
        if global_start_time is not None and time_limit is not None:
            if time.time() - global_start_time >= time_limit:
                print("Global time limit reached during neighbour search.")
                return None, None

        candidate_solution = evaluate_station_set_stochastic_greedy(
            open_station_set=neighbour_set,
            robot_ids=robot_ids,
            stochastic_distance_table=stochastic_distance_table,
            candidate_locations=candidate_locations,
            q=q,
            m=m,
            c_b=c_b,
            c_m=c_m
        )

        if candidate_solution is None:
            continue

        if candidate_solution["total_cost"] < current_cost:
            return (move_type, j_out, j_in), candidate_solution

    return None, None


In [ ]:
### Run Local Search

local_search_history = [current_solution["total_cost"]]

time_limit = 300
start_time = time.time()
iteration = 0

while True:
    iteration += 1

    move, improving_solution = find_first_improving_neighbour_stochastic(
        current_solution=current_solution,
        stochastic_distance_table=stochastic_distance_table,
        candidate_locations=candidate_locations,
        q=q,
        m=m,
        c_b=c_b,
        c_m=c_m,
        swap_nearest_k=10,
        top_robot_count=10,
        open_nearest_candidates_per_robot=5,
        max_close_candidates=5,
        allow_open=True,
        allow_close=True,
        global_start_time=start_time,
        time_limit=time_limit
    )

    if move is None:
        print("No improving neighbour found or time limit reached. Local search stops.")
        break

    current_solution = improving_solution
    local_search_history.append(current_solution["total_cost"])

    move_type, j_out, j_in = move
    print(
        f"Iteration {iteration}: {move_type} move, "
        f"remove={j_out}, add={j_in}, "
        f"cost={current_solution['total_cost']:,.2f}"
    )

final_ls_solution = current_solution

print("\nFinal local-search cost:", final_ls_solution["total_cost"])
display(final_ls_solution["stations_df"])
display(final_ls_solution["assignments_df"].head())


In [ ]:
ls_stations_df = final_ls_solution["stations_df"].copy()
ls_assignments_df = final_ls_solution["assignments_df"].copy()

ls_buildings_cost = final_ls_solution["buildings_cost"]
ls_assignment_cost = final_ls_solution["assignment_cost"]
ls_transport_cost = final_ls_solution["transport_cost"]
ls_total_cost = final_ls_solution["total_cost"]

ls_results_summary = pd.DataFrame([
    {"Metric": "Number of robots", "Value": f"{len(robot_ids):,}"},
    {"Metric": "Number of stations", "Value": f"{len(final_ls_solution['open_station_set']):,}"},
    {"Metric": "Number of chargers", "Value": f"{int(ls_stations_df['chargers'].sum()):,}"},
    {"Metric": "Total cost", "Value": f"{ls_total_cost:,.2f}"},
    {"Metric": "Assignment cost", "Value": f"{ls_assignment_cost:,.2f}"},
    {"Metric": "Buildings cost", "Value": f"{ls_buildings_cost:,.2f}"},
    {"Metric": "Transport cost", "Value": f"{ls_transport_cost:,.2f}"},
    {"Metric": "Robots needing transport (expected)", "Value": f"{ls_assignments_df['expected_unreachable'].sum():,.2f}"},
    {"Metric": "MIP gap", "Value": "N/A"},
    {"Metric": "MIP gap (%)", "Value": "N/A"}
])

display(ls_results_summary)


### Step 8: MIP Optimisation

In [ ]:
# Step 8: MIP after the local-search open-station set
# Fixed open stations, fixed robot-to-station assignment across scenarios, expected costs from stochastic_distance_table.

J_open = sorted(final_ls_solution["open_station_set"])
local_search_mip_robot_ids = robots_subset["robot_id"].astype(int).tolist()
local_search_mip_scenario_ids = sorted(scenario_table["scenario_id"].unique())

local_search_mip_dt_open = stochastic_distance_table[
    stochastic_distance_table["candidate_id"].isin(J_open)
].copy()

local_search_mip_expected_cost_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.assign_cost)
    for row in local_search_mip_dt_open.itertuples(index=False)
}
local_search_mip_expected_charging_cost_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.expected_charging_cost)
    for row in local_search_mip_dt_open.itertuples(index=False)
}
local_search_mip_expected_transport_cost_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.expected_transport_cost)
    for row in local_search_mip_dt_open.itertuples(index=False)
}
local_search_mip_expected_unreachable_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.expected_unreachable)
    for row in local_search_mip_dt_open.itertuples(index=False)
}
local_search_mip_expected_needs_charge_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.expected_needs_charge)
    for row in local_search_mip_dt_open.itertuples(index=False)
}
local_search_mip_distance_lookup = {
    (int(row.robot_id), int(row.candidate_id)): float(row.d_ij)
    for row in local_search_mip_dt_open.itertuples(index=False)
}

local_search_mip_model = pyo.ConcreteModel()
local_search_mip_model.I = pyo.Set(initialize=local_search_mip_robot_ids)
local_search_mip_model.J = pyo.Set(initialize=J_open)
local_search_mip_model.assign_cost = pyo.Param(
    local_search_mip_model.I,
    local_search_mip_model.J,
    initialize=local_search_mip_expected_cost_lookup
)

local_search_mip_model.z = pyo.Var(local_search_mip_model.J, domain=pyo.NonNegativeIntegers)
local_search_mip_model.x = pyo.Var(local_search_mip_model.I, local_search_mip_model.J, domain=pyo.Binary)

# Each robot is assigned to exactly one of the fixed open stations.
def local_search_mip_assign_once_rule(model, i):
    return sum(model.x[i, j] for j in model.J) == 1

local_search_mip_model.assign_once = pyo.Constraint(
    local_search_mip_model.I,
    rule=local_search_mip_assign_once_rule
)

# A station with z[j] chargers can serve at most q*z[j] assigned robots.
def local_search_mip_charger_capacity_rule(model, j):
    return sum(model.x[i, j] for i in model.I) <= q * model.z[j]

local_search_mip_model.charger_capacity = pyo.Constraint(
    local_search_mip_model.J,
    rule=local_search_mip_charger_capacity_rule
)

# Charger bounds at fixed open stations.
def local_search_mip_charger_limit_rule(model, j):
    return model.z[j] <= m

local_search_mip_model.charger_limit = pyo.Constraint(
    local_search_mip_model.J,
    rule=local_search_mip_charger_limit_rule
)

def local_search_mip_min_charger_rule(model, j):
    return model.z[j] >= 1

local_search_mip_model.min_charger = pyo.Constraint(
    local_search_mip_model.J,
    rule=local_search_mip_min_charger_rule
)

# Objective: fixed station building cost + charger cost + expected assignment cost.
def local_search_mip_objective_rule(model):
    buildings_cost = len(J_open) * c_b + sum(c_m * model.z[j] for j in model.J)
    expected_assignment_cost = sum(
        model.assign_cost[i, j] * model.x[i, j]
        for i in model.I
        for j in model.J
    )
    return buildings_cost + expected_assignment_cost

local_search_mip_model.obj = pyo.Objective(
    rule=local_search_mip_objective_rule,
    sense=pyo.minimize
)

local_search_mip_solver = pyo.SolverFactory("gurobi")
local_search_mip_solver.options["TimeLimit"] = 300
local_search_mip_results = local_search_mip_solver.solve(local_search_mip_model, tee=True)

mip_model = local_search_mip_model
mip_results = local_search_mip_results

mip_stations_df = pd.DataFrame([
    {
        "candidate_id": int(j),
        "chargers": int(round(pyo.value(local_search_mip_model.z[j])))
    }
    for j in local_search_mip_model.J
]).merge(
    candidate_locations[["candidate_id", "longitude", "latitude"]],
    on="candidate_id",
    how="left"
)

mip_assignment_rows = []
for i in local_search_mip_model.I:
    for j in local_search_mip_model.J:
        if pyo.value(local_search_mip_model.x[i, j]) > 0.5:
            mip_assignment_rows.append({
                "robot_id": int(i),
                "candidate_id": int(j),
                "d_ij": local_search_mip_distance_lookup[(i, j)],
                "expected_needs_charge": local_search_mip_expected_needs_charge_lookup[(i, j)],
                "expected_unreachable": local_search_mip_expected_unreachable_lookup[(i, j)],
                "expected_charging_cost": local_search_mip_expected_charging_cost_lookup[(i, j)],
                "expected_transport_cost": local_search_mip_expected_transport_cost_lookup[(i, j)],
                "assign_cost": local_search_mip_expected_cost_lookup[(i, j)]
            })

mip_assignments_df = (
    pd.DataFrame(mip_assignment_rows)
    .sort_values("robot_id")
    .reset_index(drop=True)
)

mip_stations_df["robots_assigned"] = mip_stations_df["candidate_id"].map(
    mip_assignments_df["candidate_id"].value_counts()
).fillna(0).astype(int)

mip_buildings_cost = len(J_open) * c_b + sum(
    c_m * pyo.value(local_search_mip_model.z[j])
    for j in local_search_mip_model.J
)
mip_expected_assignment_cost = sum(
    pyo.value(local_search_mip_model.assign_cost[i, j]) * pyo.value(local_search_mip_model.x[i, j])
    for i in local_search_mip_model.I
    for j in local_search_mip_model.J
)
mip_expected_recourse_cost = mip_expected_assignment_cost
mip_expected_charging_cost = mip_assignments_df["expected_charging_cost"].sum()
mip_expected_transport_cost = mip_assignments_df["expected_transport_cost"].sum()
mip_expected_transport_robots = mip_assignments_df["expected_unreachable"].sum()
mip_total_cost = pyo.value(local_search_mip_model.obj)

local_search_mip_summary_df = pd.DataFrame([
    {"Solution": "MIP after local search", "Metric": "Number of robots", "Value": f"{len(local_search_mip_robot_ids):,.0f}"},
    {"Solution": "MIP after local search", "Metric": "Number of scenarios used for expectations", "Value": f"{len(local_search_mip_scenario_ids):,.0f}"},
    {"Solution": "MIP after local search", "Metric": "Number of stations", "Value": f"{len(J_open):,.0f}"},
    {"Solution": "MIP after local search", "Metric": "Number of chargers", "Value": f"{int(round(sum(pyo.value(local_search_mip_model.z[j]) for j in local_search_mip_model.J))):,.0f}"},
    {"Solution": "MIP after local search", "Metric": "Total cost", "Value": f"{mip_total_cost:,.2f}"},
    {"Solution": "MIP after local search", "Metric": "Buildings cost", "Value": f"{mip_buildings_cost:,.2f}"},
    {"Solution": "MIP after local search", "Metric": "Expected assignment cost", "Value": f"{mip_expected_assignment_cost:,.2f}"},
    {"Solution": "MIP after local search", "Metric": "Expected charging cost", "Value": f"{mip_expected_charging_cost:,.2f}"},
    {"Solution": "MIP after local search", "Metric": "Expected transport cost", "Value": f"{mip_expected_transport_cost:,.2f}"},
    {"Solution": "MIP after local search", "Metric": "Expected robots needing transport", "Value": f"{mip_expected_transport_robots:,.2f}"},
])

local_search_mip_solution = {
    "name": "MIP after local search",
    "model": local_search_mip_model,
    "results": local_search_mip_results,
    "open_station_set": set(J_open),
    "stations_df": mip_stations_df,
    "assignments_df": mip_assignments_df,
    "summary_df": local_search_mip_summary_df,
    "total_cost": mip_total_cost,
    "buildings_cost": mip_buildings_cost,
    "expected_assignment_cost": mip_expected_assignment_cost,
    "expected_recourse_cost": mip_expected_recourse_cost,
    "expected_charging_cost": mip_expected_charging_cost,
    "expected_transport_cost": mip_expected_transport_cost,
    "expected_transport_robots": mip_expected_transport_robots,
}

print("MIP after local search objective value:", f"{mip_total_cost:,.2f}")
display(mip_stations_df)
display(mip_assignments_df.head())
display(local_search_mip_summary_df)

### Stage Metrics Summary

In [ ]:
# Metrics after each stage

def _as_float(value):
    try:
        if value is None:
            return None
        return float(value)
    except (TypeError, ValueError):
        return None


def extract_mip_gap_percent(results):
    gap = _as_float(getattr(results.solver, "mip_gap", None))

    if gap is None:
        gap = _as_float(getattr(results.solver, "gap", None))

    if gap is None:
        lower_bound = _as_float(getattr(results.problem, "lower_bound", None))
        upper_bound = _as_float(getattr(results.problem, "upper_bound", None))

        if lower_bound is not None and upper_bound is not None:
            gap = abs(upper_bound - lower_bound) / max(abs(upper_bound), 1e-9)

    if gap is None:
        return np.nan

    # Most solvers report MIP gap as a fraction, e.g. 0.01 = 1%.
    return 100 * gap


stage_metrics = pd.DataFrame([
    {
        "Stage": "Post heuristic",
        "Number of robots": len(robot_ids),
        "Number of scenarios": len(selected_scenarios),
        "Number of stations": len(heuristic_stations_df),
        "Number of chargers": int(heuristic_stations_df["chargers"].sum()),
        "Total cost": heuristic_total_cost,
        "Assignment cost": heuristic_assignment_cost,
        "Buildings cost": heuristic_buildings_cost,
        "Transport cost": heuristic_transport_cost,
        "Robots needing transport": heuristic_assignments_df["expected_unreachable"].sum(),
        "MIP gap (%)": np.nan,
    },
    {
        "Stage": "Post MIP 1",
        "Number of robots": len(robot_ids),
        "Number of scenarios": len(selected_scenarios),
        "Number of stations": len(heuristic_mip_solution["open_station_set"]),
        "Number of chargers": int(heuristic_mip_stations_df["chargers"].sum()),
        "Total cost": heuristic_mip_solution["total_cost"],
        "Assignment cost": heuristic_mip_solution["expected_charging_cost"],
        "Buildings cost": heuristic_mip_solution["buildings_cost"],
        "Transport cost": heuristic_mip_solution["expected_transport_cost"],
        "Robots needing transport": heuristic_mip_solution["expected_transport_robots"],
        "MIP gap (%)": extract_mip_gap_percent(heuristic_mip_solution["results"]),
    },
    {
        "Stage": "Post local search",
        "Number of robots": len(robot_ids),
        "Number of scenarios": len(selected_scenarios),
        "Number of stations": len(ls_stations_df),
        "Number of chargers": int(ls_stations_df["chargers"].sum()),
        "Total cost": ls_total_cost,
        "Assignment cost": ls_assignment_cost,
        "Buildings cost": ls_buildings_cost,
        "Transport cost": ls_transport_cost,
        "Robots needing transport": ls_assignments_df["expected_unreachable"].sum(),
        "MIP gap (%)": np.nan,
    },
    {
        "Stage": "Post MIP 2",
        "Number of robots": len(robot_ids),
        "Number of scenarios": len(selected_scenarios),
        "Number of stations": len(local_search_mip_solution["open_station_set"]),
        "Number of chargers": int(mip_stations_df["chargers"].sum()),
        "Total cost": mip_total_cost,
        "Assignment cost": mip_expected_charging_cost,
        "Buildings cost": mip_buildings_cost,
        "Transport cost": mip_expected_transport_cost,
        "Robots needing transport": mip_expected_transport_robots,
        "MIP gap (%)": extract_mip_gap_percent(local_search_mip_solution["results"]),
    },
])

display(
    stage_metrics.style.format({
        "Number of robots": "{:,.0f}",
        "Number of scenarios": "{:,.0f}",
        "Number of stations": "{:,.0f}",
        "Number of chargers": "{:,.0f}",
        "Total cost": "{:,.2f}",
        "Assignment cost": "{:,.2f}",
        "Buildings cost": "{:,.2f}",
        "Transport cost": "{:,.2f}",
        "Robots needing transport": "{:,.2f}",
        "MIP gap (%)": "{:,.4f}%",
    }, na_rep="N/A")
)

### Comparison Graphs

In [ ]:
# Plotting helper for assignment graphs

def plot_expected_assignment_solution(ax, stations_df, assignments_df, title):
    station_col = "candidate_id"

    transport_robot_ids = set(
        assignments_df.loc[assignments_df["expected_unreachable"] > 0, "robot_id"]
    )

    reachable_robots = robots_subset[~robots_subset["robot_id"].isin(transport_robot_ids)]
    transport_robots = robots_subset[robots_subset["robot_id"].isin(transport_robot_ids)]

    ax.scatter(
        reachable_robots["longitude"],
        reachable_robots["latitude"],
        c="red",
        s=30,
        alpha=0.65,
        label="Robots"
    )

    if not transport_robots.empty:
        ax.scatter(
            transport_robots["longitude"],
            transport_robots["latitude"],
            c="darkorange",
            s=45,
            alpha=0.8,
            label="Expected transport > 0"
        )

    ax.scatter(
        stations_df["longitude"],
        stations_df["latitude"],
        c="blue",
        s=120,
        marker="X",
        label="Stations"
    )

    for _, row in assignments_df.iterrows():
        robot = robots_subset[robots_subset["robot_id"] == row["robot_id"]].iloc[0]
        station = stations_df[stations_df[station_col] == row["candidate_id"]].iloc[0]
        line_color = "darkorange" if row["expected_unreachable"] > 0 else "gray"
        ax.plot(
            [robot["longitude"], station["longitude"]],
            [robot["latitude"], station["latitude"]],
            color=line_color,
            alpha=0.35,
            linewidth=1
        )

    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(True, alpha=0.3)


def collapse_mip_assignments_to_robot_station(mip_assignments_df):
    # The corrected expected-cost MIP already has one fixed station per robot.
    if "expected_unreachable" in mip_assignments_df.columns:
        return mip_assignments_df[["robot_id", "candidate_id", "expected_unreachable", "assign_cost"]].copy()

    # Fallback for old scenario-specific outputs, if an old cell has not been rerun yet.
    if mip_assignments_df.empty:
        return pd.DataFrame(columns=["robot_id", "candidate_id", "expected_unreachable", "assign_cost"])

    rows = []
    for robot_id, group in mip_assignments_df.groupby("robot_id"):
        station_counts = group["candidate_id"].value_counts()
        candidate_id = int(station_counts.index[0])
        chosen = group[group["candidate_id"] == candidate_id]

        rows.append({
            "robot_id": int(robot_id),
            "candidate_id": candidate_id,
            "expected_unreachable": float(chosen["unreachable"].mean()),
            "assign_cost": float(chosen["scenario_assign_cost"].mean())
        })

    return pd.DataFrame(rows)

In [ ]:
# Compare heuristic and local-search allocation graphs

fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharex=True, sharey=True)

plot_expected_assignment_solution(
    axes[0],
    heuristic_stations_df,
    heuristic_assignments_df,
    f"Heuristic allocation\nExpected cost = {heuristic_total_cost:,.2f}"
)

plot_expected_assignment_solution(
    axes[1],
    ls_stations_df,
    ls_assignments_df,
    f"After local search\nExpected cost = {ls_total_cost:,.2f}"
)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)
fig.suptitle("Heuristic vs Local Search Assignments", fontsize=14)
plt.xticks(rotation=90)
plt.tight_layout(rect=[0, 0.06, 1, 0.95])
fig.savefig("Graphs/assignments_2_compare_heur_500.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Compare MIP-after-heuristic and MIP-after-local-search allocation graphs

mip_after_heuristic_plot_assignments = collapse_mip_assignments_to_robot_station(
    heuristic_mip_assignments_df
)

mip_after_local_search_plot_assignments = collapse_mip_assignments_to_robot_station(
    mip_assignments_df
)

fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharex=True, sharey=True)

plot_expected_assignment_solution(
    axes[0],
    heuristic_mip_stations_df,
    mip_after_heuristic_plot_assignments,
    f"MIP after heuristic\nExpected cost = {heuristic_mip_total_cost:,.2f}"
)

plot_expected_assignment_solution(
    axes[1],
    mip_stations_df,
    mip_after_local_search_plot_assignments,
    f"MIP after local search\nExpected cost = {mip_total_cost:,.2f}"
)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)
fig.suptitle("MIP After Heuristic vs MIP After Local Search", fontsize=14)
plt.xticks(rotation=90)
plt.tight_layout(rect=[0, 0.06, 1, 0.95])
fig.savefig("Graphs/assignments_2_compare_MIP_500.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Plot value change across local-search iterations

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(local_search_history)), local_search_history, marker="o")
ax.set_xlabel("Local-search iteration")
ax.set_ylabel("Expected total cost")
ax.set_title("Local Search Improvement History")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig("Graphs/assignments_2_local_search_500.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Check deterministic ranges against mean ranges from all 100 range scenarios

deterministic_ranges = pd.read_csv("range.csv")
deterministic_ranges = deterministic_ranges.rename(
    columns={deterministic_ranges.columns[0]: "robot_id", "range": "deterministic_range"}
)

all_100_scenario_cols = [c for c in ranges.columns if c.startswith("s")]

mean_scenario_ranges = ranges[["robot_id"]].copy()
mean_scenario_ranges["mean_scenario_range"] = ranges[all_100_scenario_cols].mean(axis=1)
mean_scenario_ranges["std_scenario_range"] = ranges[all_100_scenario_cols].std(axis=1)

range_comparison = deterministic_ranges.merge(
    mean_scenario_ranges,
    on="robot_id",
    how="inner"
)

range_comparison["difference"] = (
    range_comparison["mean_scenario_range"]
    - range_comparison["deterministic_range"]
)
range_comparison["absolute_difference"] = range_comparison["difference"].abs()

print("Robots compared:", len(range_comparison))
print("Scenario columns used:", len(all_100_scenario_cols))

print("\nDifference summary: mean scenario range - deterministic range")
display(range_comparison["difference"].describe().to_frame())

print("\nLargest absolute differences")
display(
    range_comparison
    .sort_values("absolute_difference", ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(
    range_comparison["deterministic_range"],
    range_comparison["mean_scenario_range"],
    alpha=0.65,
    s=25
)

min_range = min(
    range_comparison["deterministic_range"].min(),
    range_comparison["mean_scenario_range"].min()
)
max_range = max(
    range_comparison["deterministic_range"].max(),
    range_comparison["mean_scenario_range"].max()
)
ax.plot([min_range, max_range], [min_range, max_range], color="black", linestyle="--", linewidth=1)

ax.set_xlabel("Deterministic ranges in Q1")
ax.set_ylabel("Mean range across 100 scenarios")
ax.set_title("Deterministic Range vs Mean Scenario Range")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()